# Melanoma experiments

Settings on the right: **Accelerator = GPU P100**, **Internet = On**.

Resumable. If the session dies, open it again and Run All; it skips whatever
already finished.


In [ ]:
import os, shutil, subprocess, time, glob
t0 = time.time()

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings > Accelerator > GPU P100, then Run All again.")
print("gpu:", torch.cuda.get_device_name(0))

try:
    import albumentations as A
    A.Affine
    print("albumentations", A.__version__)
except Exception:
    subprocess.run(["pip","install","-q","albumentations>=2.0"], check=False)
    import albumentations as A
    print("albumentations installed:", A.__version__)


In [ ]:
# --- locate the dataset -------------------------------------------------
# Kaggle unpacked melanoma_512.tar for us, so the photos are already in a
# melanoma_512/ folder. No untarring needed.
INPUT = None
for d in sorted(os.listdir("/kaggle/input")):
    if os.path.exists(f"/kaggle/input/{d}/folds.csv"):
        INPUT = f"/kaggle/input/{d}"
        break
assert INPUT, "Attach the melanoma-512-preprocessed dataset (Add Data on the right)."
print("input:", INPUT)

# train_512 may sit at the root or one level down inside melanoma_512/
IMG_DIR = None
for cand in [f"{INPUT}/train_512", f"{INPUT}/melanoma_512/train_512"]:
    if os.path.isdir(cand):
        IMG_DIR = cand
        break
assert IMG_DIR, f"train_512 not found under {INPUT}"

FOLDS = f"{INPUT}/folds.csv"
print("img_dir  :", IMG_DIR)
print("folds    :", FOLDS)
print("photos   :", len(os.listdir(IMG_DIR)))


In [ ]:
# --- copy our source files somewhere importable -------------------------
WORK = "/kaggle/working"
SRC = f"{WORK}/src"
os.makedirs(SRC, exist_ok=True)

found = 0
for path in glob.glob(f"{INPUT}/*.py") + glob.glob(f"{INPUT}/src/*.py"):
    shutil.copy(path, SRC)
    found += 1
print("copied", found, "source files:", sorted(os.listdir(SRC)))
assert found >= 5, "source .py files missing from the dataset"


In [ ]:
# --- run the programme ---------------------------------------------------
# Budget leaves headroom inside Kaggle's session limit so the run stops
# cleanly and saves, rather than being killed part way through a fold.
BUDGET = 7.5 - (time.time() - t0) / 3600
print(f"time budget: {BUDGET:.2f} h")

!cd {SRC} && python experiment_runner.py \
    --img_dir {IMG_DIR} \
    --folds_csv {FOLDS} \
    --out_dir {WORK}/reports \
    --batch_size 64 \
    --num_workers 2 \
    --time_budget_h {BUDGET:.2f}


In [ ]:
# --- report --------------------------------------------------------------
!cd {SRC} && python report_results.py --in_dir {WORK}/reports

import pandas as pd
pd.read_csv(f"{WORK}/reports/results.csv")
